In [ ]:
# ======================================================================
# STEP W2
# STRICTLY CAUSAL WEATHER FEATURE ENGINEERING
# + MERGE WITH FROZEN 165-FEATURE FORECASTING DATASET
#
# PURPOSE
# -------
# Add historical weather information WITHOUT:
#   - changing Steps 1-8
#   - changing target dates
#   - changing target values
#   - using target-day realized weather
#   - using future weather
#
# Forecast origin:
#   After 23:00 on day D-1
#
# Therefore for target day D:
#   Weather from D-1 and earlier is allowed.
#   Actual weather from D is NOT used.
#
# OUTPUT:
#
# data/forecasting_weather/
#   train_2015_2022_weather.csv
#   validation_2023_weather.csv
#   test_2024_2026_weather.csv
#   weather_features_by_target_date.csv
#   weather_feature_manifest.csv
#
# results/weather_extension/
#   stepW2_merge_audit.csv
#
# IMPORTANT:
#   Missing weather values are NOT imputed here.
#   Imputation will be fitted ONLY on training data in Step W3.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 300)


print("=" * 120)
print("STEP W2: STRICTLY CAUSAL WEATHER FEATURE ENGINEERING")
print("=" * 120)


# ======================================================================
# 1. MOUNT GOOGLE DRIVE
# ======================================================================

drive.mount(
    "/content/drive"
)


# ======================================================================
# 2. PROJECT PATHS
# ======================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Load_Forecasting_Paper"
)

FORECAST_DIR = (
    PROJECT_ROOT
    / "data"
    / "forecasting"
)

WEATHER_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "weather"
    / "processed"
)

WEATHER_FORECAST_DIR = (
    PROJECT_ROOT
    / "data"
    / "forecasting_weather"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "weather_extension"
)


for directory in [

    WEATHER_FORECAST_DIR,
    RESULT_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ======================================================================
# 3. INPUT FILES
# ======================================================================

WEATHER_FILE = (
    WEATHER_PROCESSED_DIR
    / "bangladesh_national_hourly_weather.csv"
)


TRAIN_FILE = (
    FORECAST_DIR
    / "train_2015_2022.csv"
)


VALIDATION_FILE = (
    FORECAST_DIR
    / "validation_2023.csv"
)


TEST_FILE = (
    FORECAST_DIR
    / "test_2024_2026.csv"
)


required_files = [

    WEATHER_FILE,
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE

]


print("\nRequired files:")

for path in required_files:

    print(
        path,
        "->",
        path.exists()
    )

    if not path.exists():

        raise FileNotFoundError(
            f"""
Required file not found:

{path}
"""
        )


print(
    "\nAll required files found."
)


# ======================================================================
# 4. LOAD NATIONAL HOURLY WEATHER
# ======================================================================

weather = pd.read_csv(
    WEATHER_FILE,
    low_memory=False
)


weather["datetime"] = pd.to_datetime(
    weather["datetime"],
    errors="coerce"
)


weather = (

    weather
    .dropna(
        subset=["datetime"]
    )

    .sort_values(
        "datetime"
    )

    .reset_index(
        drop=True
    )
)


weather["date"] = (
    weather["datetime"]
    .dt.normalize()
)


weather["hour"] = (
    weather["datetime"]
    .dt.hour
)


print("\n" + "=" * 120)
print("NATIONAL WEATHER INPUT")
print("=" * 120)


print(
    "Rows:",
    f"{len(weather):,}"
)

print(
    "Start:",
    weather["datetime"].min()
)

print(
    "End:",
    weather["datetime"].max()
)


# ======================================================================
# 5. VALIDATE HOURLY WEATHER STRUCTURE
# ======================================================================

duplicate_timestamps = int(
    weather[
        "datetime"
    ].duplicated().sum()
)


print(
    "Duplicate hourly timestamps:",
    duplicate_timestamps
)


if duplicate_timestamps > 0:

    raise ValueError(
        "Duplicate national weather timestamps found."
    )


full_hourly_range = pd.date_range(

    start=
        weather["datetime"].min(),

    end=
        weather["datetime"].max(),

    freq="h"
)


missing_hourly_timestamps = (
    full_hourly_range
    .difference(
        pd.DatetimeIndex(
            weather["datetime"]
        )
    )
)


print(
    "Missing hourly timestamps:",
    len(
        missing_hourly_timestamps
    )
)


if len(
    missing_hourly_timestamps
) > 0:

    raise ValueError(
        "National hourly weather is not continuous."
    )


# ======================================================================
# 6. DEFINE DAILY WEATHER SUMMARIES
# ======================================================================
#
# These summarize observed weather for each historical day.
#
# For target day D, only the summary from D-1 or earlier
# will later be used.
# ======================================================================

daily = (

    weather
    .groupby(
        "date"
    )
    .agg(

        temp_mean_d=(
            "temp_mean",
            "mean"
        ),

        temp_max_d=(
            "temp_max",
            "max"
        ),

        temp_min_d=(
            "temp_min",
            "min"
        ),

        humidity_mean_d=(
            "humidity_mean",
            "mean"
        ),

        humidity_max_d=(
            "humidity_max",
            "max"
        ),

        rain_total_d=(
            "rain_mean",
            "sum"
        ),

        rain_peak_d=(
            "rain_max",
            "max"
        ),

        rain_city_fraction_d=(
            "rain_city_fraction",
            "mean"
        ),

        pressure_mean_d=(
            "pressure_mean",
            "mean"
        ),

        cloud_mean_d=(
            "cloud_mean",
            "mean"
        ),

        wind_mean_d=(
            "wind_mean",
            "mean"
        ),

        wind_max_d=(
            "wind_max",
            "max"
        ),

        temp_humidity_mean_d=(
            "temp_humidity_mean",
            "mean"
        ),

        spatial_temp_range_mean_d=(
            "temp_range_spatial",
            "mean"
        )
    )
)


# ======================================================================
# 7. FORCE COMPLETE CALENDAR
# ======================================================================
#
# This is important.
#
# We use calendar-day shifts rather than "previous dataframe row",
# so D-1 really means previous calendar day.
# ======================================================================

full_daily_index = pd.date_range(

    start=
        daily.index.min(),

    end=
        daily.index.max(),

    freq="D"
)


daily = daily.reindex(
    full_daily_index
)


daily.index.name = "date"


print("\nDaily weather days:")
print(
    f"{len(daily):,}"
)


print(
    "Daily period:",
    daily.index.min(),
    "to",
    daily.index.max()
)


# ======================================================================
# 8. PREVIOUS-DAY DAILY WEATHER
# ======================================================================
#
# target D receives weather from D-1.
# ======================================================================

lag1_daily = (
    daily
    .shift(1)
    .copy()
)


lag1_daily.columns = [

    f"wx_lag1_{column}"

    for column
    in lag1_daily.columns
]


# ======================================================================
# 9. PREVIOUS-DAY HOURLY TEMPERATURE PROFILE
# ======================================================================
#
# These 24 values describe the complete temperature profile of D-1.
#
# Example:
#   target D
#     wx_lag1_temp_h00 = temperature at D-1 00:00
#     ...
#     wx_lag1_temp_h23 = temperature at D-1 23:00
#
# All are available at the defined forecast origin.
# ======================================================================

temp_hourly = (

    weather
    .pivot(
        index="date",
        columns="hour",
        values="temp_mean"
    )

    .reindex(
        full_daily_index
    )
)


temp_hourly = (
    temp_hourly
    .reindex(
        columns=
            list(
                range(24)
            )
    )
)


temp_hourly.columns = [

    f"wx_lag1_temp_h{hour:02d}"

    for hour
    in range(24)
]


temp_hourly = (
    temp_hourly
    .shift(1)
)


# ======================================================================
# 10. PREVIOUS-DAY HOURLY HUMIDITY PROFILE
# ======================================================================

humidity_hourly = (

    weather
    .pivot(
        index="date",
        columns="hour",
        values="humidity_mean"
    )

    .reindex(
        full_daily_index
    )
)


humidity_hourly = (
    humidity_hourly
    .reindex(
        columns=
            list(
                range(24)
            )
    )
)


humidity_hourly.columns = [

    f"wx_lag1_humidity_h{hour:02d}"

    for hour
    in range(24)
]


humidity_hourly = (
    humidity_hourly
    .shift(1)
)


# ======================================================================
# 11. ROLLING WEATHER VARIABLES
# ======================================================================

ROLLING_MEAN_VARIABLES = [

    "temp_mean_d",
    "temp_max_d",
    "humidity_mean_d",
    "rain_total_d",
    "cloud_mean_d",
    "wind_mean_d",
    "pressure_mean_d",
    "temp_humidity_mean_d"

]


ROLLING_STD_VARIABLES = [

    "temp_mean_d",
    "humidity_mean_d",
    "rain_total_d",
    "temp_humidity_mean_d"

]


# ======================================================================
# 12. 7-DAY ROLLING MEANS
# ======================================================================
#
# shift(1) is applied AFTER rolling.
#
# Therefore target D never uses weather from D itself.
# ======================================================================

roll7_mean = (

    daily[
        ROLLING_MEAN_VARIABLES
    ]

    .rolling(
        window=7,
        min_periods=4
    )

    .mean()

    .shift(1)
)


roll7_mean.columns = [

    f"wx_roll7mean_{column}"

    for column
    in roll7_mean.columns
]


# ======================================================================
# 13. 28-DAY ROLLING MEANS
# ======================================================================

roll28_mean = (

    daily[
        ROLLING_MEAN_VARIABLES
    ]

    .rolling(
        window=28,
        min_periods=14
    )

    .mean()

    .shift(1)
)


roll28_mean.columns = [

    f"wx_roll28mean_{column}"

    for column
    in roll28_mean.columns
]


# ======================================================================
# 14. 7-DAY WEATHER VARIABILITY
# ======================================================================

roll7_std = (

    daily[
        ROLLING_STD_VARIABLES
    ]

    .rolling(
        window=7,
        min_periods=4
    )

    .std()

    .shift(1)
)


roll7_std.columns = [

    f"wx_roll7std_{column}"

    for column
    in roll7_std.columns
]


# ======================================================================
# 15. 28-DAY WEATHER VARIABILITY
# ======================================================================

roll28_std = (

    daily[
        ROLLING_STD_VARIABLES
    ]

    .rolling(
        window=28,
        min_periods=14
    )

    .std()

    .shift(1)
)


roll28_std.columns = [

    f"wx_roll28std_{column}"

    for column
    in roll28_std.columns
]


# ======================================================================
# 16. HISTORICAL WEATHER AVAILABILITY COUNTS
# ======================================================================

daily_observed = (
    daily[
        "temp_mean_d"
    ]
    .notna()
    .astype(int)
)


history7_count = (

    daily_observed
    .rolling(
        window=7,
        min_periods=1
    )

    .sum()

    .shift(1)

    .rename(
        "wx_history7_days_available"
    )
)


history28_count = (

    daily_observed
    .rolling(
        window=28,
        min_periods=1
    )

    .sum()

    .shift(1)

    .rename(
        "wx_history28_days_available"
    )
)


# ======================================================================
# 17. D-1 WEATHER AVAILABILITY FLAG
# ======================================================================

lag1_available = (

    daily[
        "temp_mean_d"
    ]

    .shift(1)

    .notna()

    .astype(int)

    .rename(
        "wx_lag1_available"
    )
)


# ======================================================================
# 18. COMBINE ALL CAUSAL WEATHER FEATURES
# ======================================================================

weather_features = pd.concat(

    [

        lag1_daily,

        temp_hourly,

        humidity_hourly,

        roll7_mean,

        roll28_mean,

        roll7_std,

        roll28_std,

        history7_count,

        history28_count,

        lag1_available

    ],

    axis=1
)


weather_features.index.name = (
    "target_date"
)


weather_features = (
    weather_features
    .reset_index()
)


print("\n" + "=" * 120)
print("CAUSAL WEATHER FEATURE MATRIX")
print("=" * 120)


print(
    "Rows:",
    f"{len(weather_features):,}"
)


WEATHER_FEATURE_COLUMNS = [

    column

    for column
    in weather_features.columns

    if column != "target_date"
]


print(
    "Weather feature count:",
    len(
        WEATHER_FEATURE_COLUMNS
    )
)


print(
    "Feature period:",
    weather_features[
        "target_date"
    ].min(),
    "to",
    weather_features[
        "target_date"
    ].max()
)


# ======================================================================
# 19. WEATHER FEATURE GROUPS
# ======================================================================

FEATURE_GROUPS_WEATHER = {

    "Previous_Day_Daily_Weather":
        list(
            lag1_daily.columns
        ),

    "Previous_Day_Hourly_Temperature":
        list(
            temp_hourly.columns
        ),

    "Previous_Day_Hourly_Humidity":
        list(
            humidity_hourly.columns
        ),

    "Weather_Rolling7_Mean":
        list(
            roll7_mean.columns
        ),

    "Weather_Rolling28_Mean":
        list(
            roll28_mean.columns
        ),

    "Weather_Rolling7_STD":
        list(
            roll7_std.columns
        ),

    "Weather_Rolling28_STD":
        list(
            roll28_std.columns
        ),

    "Weather_Availability":
        [
            "wx_history7_days_available",
            "wx_history28_days_available",
            "wx_lag1_available"
        ]
}


print("\nWeather feature groups:")


for group, features in FEATURE_GROUPS_WEATHER.items():

    print(
        f"{group:38s}:",
        len(features)
    )


# ======================================================================
# 20. FEATURE MANIFEST
# ======================================================================

manifest_rows = []


for group_name, columns in (
    FEATURE_GROUPS_WEATHER.items()
):

    for feature_name in columns:

        manifest_rows.append({

            "feature_group":
                group_name,

            "feature":
                feature_name,

            "causal":
                True,

            "latest_possible_information":
                "target_date - 1 day or earlier"

        })


manifest = pd.DataFrame(
    manifest_rows
)


MANIFEST_FILE = (
    WEATHER_FORECAST_DIR
    / "weather_feature_manifest.csv"
)


manifest.to_csv(
    MANIFEST_FILE,
    index=False
)


# ======================================================================
# 21. SAVE WEATHER FEATURES BY TARGET DATE
# ======================================================================

WEATHER_FEATURE_FILE = (

    WEATHER_FORECAST_DIR
    / "weather_features_by_target_date.csv"

)


weather_features.to_csv(

    WEATHER_FEATURE_FILE,

    index=False
)


print("\nSaved weather feature matrix:")
print(
    WEATHER_FEATURE_FILE
)


# ======================================================================
# 22. DEFINE FROZEN 165 FEATURES
# ======================================================================

HOURS = list(
    range(
        1,
        24
    )
)


CALENDAR_FEATURES = [

    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "quarter",
    "is_weekend",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos"

]


PREVIOUS_DAY_PROFILE = [

    f"lag1_h{hour:02d}"

    for hour
    in HOURS
]


PREVIOUS_WEEK_PROFILE = [

    f"lag7_h{hour:02d}"

    for hour
    in HOURS
]


ROLLING7_MEAN_LOAD = [

    f"roll7mean_h{hour:02d}"

    for hour
    in HOURS
]


ROLLING28_MEAN_LOAD = [

    f"roll28mean_h{hour:02d}"

    for hour
    in HOURS
]


ROLLING7_STD_LOAD = [

    f"roll7std_h{hour:02d}"

    for hour
    in HOURS
]


ROLLING28_STD_LOAD = [

    f"roll28std_h{hour:02d}"

    for hour
    in HOURS
]


HISTORICAL_SUMMARY = [

    "lag1_daily_mean",
    "lag1_daily_max",
    "lag1_daily_min",
    "lag1_daily_range",
    "lag1_peak_hour",

    "lag7_daily_mean",
    "lag7_daily_max",
    "lag7_daily_min",
    "lag7_daily_range",
    "lag7_peak_hour",

    "mean_change_vs_last_week",
    "max_change_vs_last_week",
    "mean_ratio_vs_last_week"

]


BASE_FEATURE_COLUMNS = (

    PREVIOUS_DAY_PROFILE
    +
    PREVIOUS_WEEK_PROFILE
    +
    ROLLING7_MEAN_LOAD
    +
    ROLLING28_MEAN_LOAD
    +
    ROLLING7_STD_LOAD
    +
    ROLLING28_STD_LOAD
    +
    CALENDAR_FEATURES
    +
    HISTORICAL_SUMMARY

)


assert (
    len(
        BASE_FEATURE_COLUMNS
    )
    ==
    165
)


TARGET_COLUMNS = [

    f"target_h{hour:02d}"

    for hour
    in HOURS
]


assert (
    len(
        TARGET_COLUMNS
    )
    ==
    23
)


# ======================================================================
# 23. OUTPUT PATHS
# ======================================================================

OUTPUT_FILES = {

    "train":
        WEATHER_FORECAST_DIR
        / "train_2015_2022_weather.csv",

    "validation":
        WEATHER_FORECAST_DIR
        / "validation_2023_weather.csv",

    "test":
        WEATHER_FORECAST_DIR
        / "test_2024_2026_weather.csv"

}


INPUT_FILES = {

    "train":
        TRAIN_FILE,

    "validation":
        VALIDATION_FILE,

    "test":
        TEST_FILE

}


# ======================================================================
# 24. MERGE WEATHER WITH FROZEN FORECAST DATA
# ======================================================================

audit_rows = []


for split_name in [

    "train",
    "validation",
    "test"

]:

    print(
        "\n" + "-" * 120
    )

    print(
        "PROCESSING:",
        split_name.upper()
    )

    print(
        "-" * 120
    )


    original = pd.read_csv(

        INPUT_FILES[
            split_name
        ],

        low_memory=False
    )


    original[
        "target_date"
    ] = pd.to_datetime(

        original[
            "target_date"
        ],

        errors="coerce"
    )


    original[
        "target_date"
    ] = (

        original[
            "target_date"
        ]

        .dt.normalize()
    )


    # --------------------------------------------------------------
    # Verify frozen features
    # --------------------------------------------------------------

    missing_base_features = [

        feature

        for feature
        in BASE_FEATURE_COLUMNS

        if feature
        not in original.columns
    ]


    if missing_base_features:

        raise ValueError(
            f"""
{split_name}: frozen feature columns missing:

{missing_base_features}
"""
        )


    missing_targets = [

        target

        for target
        in TARGET_COLUMNS

        if target
        not in original.columns
    ]


    if missing_targets:

        raise ValueError(
            f"""
{split_name}: target columns missing:

{missing_targets}
"""
        )


    # --------------------------------------------------------------
    # Preserve target values before merge
    # --------------------------------------------------------------

    targets_before = (

        original[
            TARGET_COLUMNS
        ]

        .to_numpy(
            dtype=float
        )

        .copy()
    )


    rows_before = len(
        original
    )


    # --------------------------------------------------------------
    # Merge
    # --------------------------------------------------------------

    merged = original.merge(

        weather_features,

        on="target_date",

        how="left",

        validate="one_to_one"
    )


    # --------------------------------------------------------------
    # Structural assertions
    # --------------------------------------------------------------

    assert (
        len(
            merged
        )
        ==
        rows_before
    )


    targets_after = (

        merged[
            TARGET_COLUMNS
        ]

        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(

        targets_before,

        targets_after,

        equal_nan=True

    ):

        raise AssertionError(
            f"{split_name}: target values changed during weather merge."
        )


    # --------------------------------------------------------------
    # Weather coverage
    # --------------------------------------------------------------

    core_weather_available = (

        merged[
            "wx_lag1_available"
        ]

        .fillna(0)
        ==
        1
    )


    all_weather_complete = (

        merged[
            WEATHER_FEATURE_COLUMNS
        ]

        .notna()

        .all(
            axis=1
        )
    )


    weather_nan_cells = int(

        merged[
            WEATHER_FEATURE_COLUMNS
        ]

        .isna()

        .sum()

        .sum()
    )


    audit_rows.append({

        "split":
            split_name,

        "rows":
            len(
                merged
            ),

        "start_date":
            merged[
                "target_date"
            ].min(),

        "end_date":
            merged[
                "target_date"
            ].max(),

        "base_feature_count":
            165,

        "weather_feature_count":
            len(
                WEATHER_FEATURE_COLUMNS
            ),

        "combined_feature_count":
            (
                165
                +
                len(
                    WEATHER_FEATURE_COLUMNS
                )
            ),

        "rows_with_Dminus1_weather":
            int(
                core_weather_available.sum()
            ),

        "Dminus1_weather_coverage_percent":
            float(
                core_weather_available.mean()
                *
                100
            ),

        "rows_complete_all_weather_features":
            int(
                all_weather_complete.sum()
            ),

        "complete_weather_feature_percent":
            float(
                all_weather_complete.mean()
                *
                100
            ),

        "weather_nan_cells":
            weather_nan_cells

    })


    # --------------------------------------------------------------
    # Save
    # --------------------------------------------------------------

    merged.to_csv(

        OUTPUT_FILES[
            split_name
        ],

        index=False
    )


    print(
        "Rows:",
        len(
            merged
        )
    )

    print(
        "D-1 weather available:",
        int(
            core_weather_available.sum()
        ),
        "/",
        len(
            merged
        ),
        f"({core_weather_available.mean()*100:.2f}%)"
    )

    print(
        "Complete weather features:",
        int(
            all_weather_complete.sum()
        ),
        "/",
        len(
            merged
        ),
        f"({all_weather_complete.mean()*100:.2f}%)"
    )

    print(
        "Saved:",
        OUTPUT_FILES[
            split_name
        ]
    )


# ======================================================================
# 25. MERGE AUDIT
# ======================================================================

audit = pd.DataFrame(
    audit_rows
)


AUDIT_FILE = (

    RESULT_DIR
    / "stepW2_merge_audit.csv"

)


audit.to_csv(

    AUDIT_FILE,

    index=False
)


print("\n" + "=" * 120)
print("STEP W2 MERGE AUDIT")
print("=" * 120)


display(
    audit
)


# ======================================================================
# 26. CAUSALITY / LEAKAGE ASSERTIONS
# ======================================================================
#
# No actual target-day weather variable should exist.
#
# Every primary weather predictor must represent:
#
#   lag1 historical weather
#
# or
#
#   rolling historical weather shifted by one day.
#
# ======================================================================

for column in WEATHER_FEATURE_COLUMNS:

    valid_prefix = (

        column.startswith(
            "wx_lag1_"
        )

        or

        column.startswith(
            "wx_roll7"
        )

        or

        column.startswith(
            "wx_roll28"
        )

        or

        column.startswith(
            "wx_history"
        )

    )


    if not valid_prefix:

        raise AssertionError(
            f"Unexpected weather feature: {column}"
        )


print(
    "\nCausal weather-feature naming check PASSED."
)


# ======================================================================
# 27. FEATURE COUNTS
# ======================================================================

print("\n" + "=" * 120)
print("FINAL FEATURE STRUCTURE")
print("=" * 120)


print(
    "Frozen load/calendar features:",
    len(
        BASE_FEATURE_COLUMNS
    )
)


print(
    "Weather features:",
    len(
        WEATHER_FEATURE_COLUMNS
    )
)


print(
    "Total candidate features:",
    len(
        BASE_FEATURE_COLUMNS
    )
    +
    len(
        WEATHER_FEATURE_COLUMNS
    )
)


print(
    "Targets:",
    len(
        TARGET_COLUMNS
    )
)


# ======================================================================
# 28. SAVE STEP SUMMARY
# ======================================================================

summary = {

    "step":
        "W2",

    "weather_source_file":
        str(
            WEATHER_FILE
        ),

    "base_feature_count":
        int(
            len(
                BASE_FEATURE_COLUMNS
            )
        ),

    "weather_feature_count":
        int(
            len(
                WEATHER_FEATURE_COLUMNS
            )
        ),

    "total_candidate_feature_count":
        int(
            len(
                BASE_FEATURE_COLUMNS
            )
            +
            len(
                WEATHER_FEATURE_COLUMNS
            )
        ),

    "target_count":
        int(
            len(
                TARGET_COLUMNS
            )
        ),

    "uses_target_day_realized_weather":
        False,

    "maximum_weather_information_date":
        "target_date - 1 day",

    "weather_missing_values_imputed":
        False,

    "imputation_protocol":
        (
            "Deferred to Step W3; "
            "imputer will be fitted on training data only."
        )

}


SUMMARY_FILE = (

    RESULT_DIR
    / "stepW2_summary.json"

)


with open(

    SUMMARY_FILE,

    "w"

) as file:

    json.dump(

        summary,

        file,

        indent=4
    )


# ======================================================================
# 29. FINAL OUTPUT
# ======================================================================

print("\n")
print("=" * 120)
print("STEP W2 COMPLETED SUCCESSFULLY")
print("=" * 120)


print(
    "\nWeather feature matrix:"
)

print(
    WEATHER_FEATURE_FILE
)


print(
    "\nFeature manifest:"
)

print(
    MANIFEST_FILE
)


print(
    "\nTrain weather dataset:"
)

print(
    OUTPUT_FILES[
        "train"
    ]
)


print(
    "\nValidation weather dataset:"
)

print(
    OUTPUT_FILES[
        "validation"
    ]
)


print(
    "\nTest weather dataset:"
)

print(
    OUTPUT_FILES[
        "test"
    ]
)


print(
    "\nMerge audit:"
)

print(
    AUDIT_FILE
)


print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)


print("\nNEXT STEP: W3")
print(
    "Training-only weather imputation + scaling + "
    "Ridge baseline vs Ridge+Weather comparison."
)

print("=" * 120)

STEP W2: STRICTLY CAUSAL WEATHER FEATURE ENGINEERING
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Required files:
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/processed/bangladesh_national_hourly_weather.csv -> True
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting/train_2015_2022.csv -> True
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting/validation_2023.csv -> True
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting/test_2024_2026.csv -> True

All required files found.

NATIONAL WEATHER INPUT
Rows: 95,880
Start: 2015-04-01 00:00:00
End: 2026-03-08 23:00:00
Duplicate hourly timestamps: 0
Missing hourly timestamps: 0

Daily weather days:
3,995
Daily period: 2015-04-01 00:00:00 to 2026-03-08 00:00:00

CAUSAL WEATHER FEATURE MATRIX
Rows: 3,995
Weather feature count: 89
Feature period: 2015-04-01 00:00:00 to 2026-03-08 00:00:00

Weather feature groups:


,split,rows,start_date,end_date,base_feature_count,weather_feature_count,combined_feature_count,rows_with_Dminus1_weather,Dminus1_weather_coverage_percent,rows_complete_all_weather_features,complete_weather_feature_percent,weather_nan_cells
0,train,2236,2015-05-11,2022-12-31,165,89,254,2236,100.0,2236,100.0,0
1,validation,273,2023-01-01,2023-12-28,165,89,254,273,100.0,273,100.0,0
2,test,709,2024-01-01,2026-03-08,165,89,254,709,100.0,709,100.0,0



Causal weather-feature naming check PASSED.

FINAL FEATURE STRUCTURE
Frozen load/calendar features: 165
Weather features: 89
Total candidate features: 254
Targets: 23


STEP W2 COMPLETED SUCCESSFULLY

Weather feature matrix:
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting_weather/weather_features_by_target_date.csv

Feature manifest:
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting_weather/weather_feature_manifest.csv

Train weather dataset:
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting_weather/train_2015_2022_weather.csv

Validation weather dataset:
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting_weather/validation_2023_weather.csv

Test weather dataset:
/content/drive/MyDrive/Load_Forecasting_Paper/data/forecasting_weather/test_2024_2026_weather.csv

Merge audit:
/content/drive/MyDrive/Load_Forecasting_Paper/results/weather_extension/stepW2_merge_audit.csv

Summary:
/content/drive/MyDrive/Load_Forecasting_Paper/results/wea

In [ ]:
# ======================================================================
# STEP W0
# DOWNLOAD CONSISTENT ERA5 WEATHER DATA FOR FULL STUDY PERIOD
#
# Source:
#   Open-Meteo Historical Weather API
#
# Reanalysis:
#   ERA5
#
# Download period:
#   2015-04-01 -> 2026-03-08
#
# Why start 2015-04-01?
#   The forecasting dataset starts in May 2015 and W2 uses
#   historical 28-day rolling weather features.
#
# Cities:
#   Dhaka
#   Chittagong
#   Rajshahi
#   Sylhet
#
# Output schema matches the weather dataset already used in W1:
#
#   time
#   temperature_2m
#   relative_humidity_2m
#   rain
#   pressure_msl
#   cloud_cover
#   wind_speed_100m
#   wind_direction_100m
#   soil_temperature_100_to_255cm
#   city
#
# FINAL OUTPUT:
# /content/drive/MyDrive/Load_Forecasting_Paper/data/weather/raw/
# bangladesh_weather_era5_2015_2026.csv
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import time
import json
import requests
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)


print("=" * 115)
print("STEP W0: DOWNLOAD FULL-PERIOD ERA5 WEATHER DATA")
print("=" * 115)


# ======================================================================
# 1. MOUNT GOOGLE DRIVE
# ======================================================================

drive.mount("/content/drive")


# ======================================================================
# 2. DIRECTORIES
# ======================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Load_Forecasting_Paper"
)

WEATHER_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "weather"
    / "raw"
)

WEATHER_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_FILE = (
    WEATHER_RAW_DIR
    / "bangladesh_weather_era5_2015_2026.csv"
)


# ======================================================================
# 3. CITY COORDINATES
# ======================================================================

CITIES = {

    "Dhaka": {
        "latitude": 23.8103,
        "longitude": 90.4125
    },

    "Chittagong": {
        "latitude": 22.3569,
        "longitude": 91.7832
    },

    "Rajshahi": {
        "latitude": 24.3745,
        "longitude": 88.6042
    },

    "Sylhet": {
        "latitude": 24.8949,
        "longitude": 91.8687
    }

}


# ======================================================================
# 4. DOWNLOAD PERIOD
# ======================================================================

START_DATE = "2015-04-01"
END_DATE   = "2026-03-08"


# ======================================================================
# 5. OPEN-METEO SETTINGS
# ======================================================================

API_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)


HOURLY_VARIABLES = [

    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "pressure_msl",
    "cloud_cover",
    "wind_speed_100m",
    "wind_direction_100m",
    "soil_temperature_100_to_255cm"

]


# ======================================================================
# 6. DOWNLOAD FUNCTION
# ======================================================================
#
# Download one calendar year at a time.
#
# This is intentionally more robust than requesting the entire
# 11-year period in one very large request.
# ======================================================================

def download_weather_chunk(
    city,
    latitude,
    longitude,
    start_date,
    end_date,
    max_retries=5
):

    params = {

        "latitude":
            latitude,

        "longitude":
            longitude,

        "start_date":
            start_date,

        "end_date":
            end_date,

        "hourly":
            ",".join(
                HOURLY_VARIABLES
            ),

        # Use a single consistent reanalysis model.
        "models":
            "era5",

        # Important:
        # PGCB timestamps are interpreted in Bangladesh local time.
        "timezone":
            "Asia/Dhaka",

        "temperature_unit":
            "celsius",

        "wind_speed_unit":
            "kmh",

        "precipitation_unit":
            "mm",

        "timeformat":
            "iso8601"
    }


    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = requests.get(
                API_URL,
                params=params,
                timeout=120
            )


            if response.status_code == 429:

                wait_seconds = (
                    10 * attempt
                )

                print(
                    f"Rate limited. Waiting {wait_seconds}s..."
                )

                time.sleep(
                    wait_seconds
                )

                continue


            response.raise_for_status()


            data = response.json()


            if "hourly" not in data:

                raise ValueError(
                    f"No hourly data returned: {data}"
                )


            hourly = data["hourly"]


            frame = pd.DataFrame({

                "time":
                    hourly["time"]

            })


            for variable in HOURLY_VARIABLES:

                if variable not in hourly:

                    raise ValueError(
                        f"""
Variable missing from API response:

{variable}

City:
{city}

Period:
{start_date} -> {end_date}
"""
                    )


                frame[
                    variable
                ] = hourly[
                    variable
                ]


            frame[
                "city"
            ] = city


            return frame


        except Exception as error:

            print(
                f"\nAttempt {attempt}/{max_retries} failed:"
            )

            print(
                error
            )


            if attempt == max_retries:

                raise


            wait_seconds = (
                5 * attempt
            )

            print(
                f"Retrying in {wait_seconds} seconds..."
            )

            time.sleep(
                wait_seconds
            )


# ======================================================================
# 7. BUILD YEARLY DOWNLOAD WINDOWS
# ======================================================================

overall_start = pd.Timestamp(
    START_DATE
)

overall_end = pd.Timestamp(
    END_DATE
)


download_windows = []


for year in range(
    overall_start.year,
    overall_end.year + 1
):

    year_start = max(
        overall_start,
        pd.Timestamp(
            f"{year}-01-01"
        )
    )

    year_end = min(
        overall_end,
        pd.Timestamp(
            f"{year}-12-31"
        )
    )


    download_windows.append(

        (
            year_start.strftime(
                "%Y-%m-%d"
            ),

            year_end.strftime(
                "%Y-%m-%d"
            )
        )
    )


print(
    "\nDownload windows:"
)

for window in download_windows:

    print(
        window[0],
        "->",
        window[1]
    )


# ======================================================================
# 8. DOWNLOAD ALL CITIES
# ======================================================================

downloaded_chunks = []


for city, coordinates in (
    CITIES.items()
):

    print(
        "\n" + "=" * 115
    )

    print(
        "CITY:",
        city
    )

    print(
        "=" * 115
    )


    for chunk_start, chunk_end in (
        download_windows
    ):

        print(
            f"\nDownloading {chunk_start} -> {chunk_end}"
        )


        chunk = download_weather_chunk(

            city=
                city,

            latitude=
                coordinates[
                    "latitude"
                ],

            longitude=
                coordinates[
                    "longitude"
                ],

            start_date=
                chunk_start,

            end_date=
                chunk_end

        )


        print(
            "Rows received:",
            f"{len(chunk):,}"
        )


        downloaded_chunks.append(
            chunk
        )


        # Be polite to the free API.
        time.sleep(
            1
        )


# ======================================================================
# 9. COMBINE
# ======================================================================

weather = pd.concat(

    downloaded_chunks,

    ignore_index=True
)


weather[
    "time"
] = pd.to_datetime(

    weather[
        "time"
    ],

    errors="coerce"
)


weather = (

    weather

    .dropna(
        subset=[
            "time"
        ]
    )

    .sort_values(
        [
            "city",
            "time"
        ]
    )

    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 115)
print("DOWNLOAD COMBINED")
print("=" * 115)


print(
    "Rows:",
    f"{len(weather):,}"
)

print(
    "Start:",
    weather[
        "time"
    ].min()
)

print(
    "End:",
    weather[
        "time"
    ].max()
)

print(
    "Cities:",
    weather[
        "city"
    ].nunique()
)


# ======================================================================
# 10. COLUMN ORDER
# ======================================================================

FINAL_COLUMNS = [

    "time",

    "temperature_2m",

    "relative_humidity_2m",

    "rain",

    "pressure_msl",

    "cloud_cover",

    "wind_speed_100m",

    "wind_direction_100m",

    "soil_temperature_100_to_255cm",

    "city"

]


weather = weather[
    FINAL_COLUMNS
]


# ======================================================================
# 11. DUPLICATE CHECK
# ======================================================================

duplicate_count = int(

    weather.duplicated(

        subset=[
            "time",
            "city"
        ]

    ).sum()
)


print(
    "\nDuplicate city-hour rows:",
    duplicate_count
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate city-hour observations detected."
    )


# ======================================================================
# 12. MISSING VALUE AUDIT
# ======================================================================

weather_variables = [

    column

    for column
    in FINAL_COLUMNS

    if column
    not in [
        "time",
        "city"
    ]
]


missing_report = pd.DataFrame({

    "missing_count":

        weather[
            weather_variables
        ]
        .isna()
        .sum(),

    "missing_percent":

        weather[
            weather_variables
        ]
        .isna()
        .mean()
        *
        100

})


print("\n" + "=" * 115)
print("MISSING VALUE AUDIT")
print("=" * 115)


display(
    missing_report
)


# ======================================================================
# 13. CITY COVERAGE
# ======================================================================

city_summary = (

    weather

    .groupby(
        "city"
    )

    .agg(

        rows=(
            "time",
            "size"
        ),

        start=(
            "time",
            "min"
        ),

        end=(
            "time",
            "max"
        ),

        unique_hours=(
            "time",
            "nunique"
        )

    )

    .reset_index()
)


print("\n" + "=" * 115)
print("CITY COVERAGE")
print("=" * 115)


display(
    city_summary
)


# ======================================================================
# 14. EXPECTED HOURLY COVERAGE
# ======================================================================

expected_hours = pd.date_range(

    start=
        pd.Timestamp(
            START_DATE
        ),

    end=
        pd.Timestamp(
            END_DATE
        )
        +
        pd.Timedelta(
            hours=23
        ),

    freq="h"
)


print(
    "\nExpected hours per city:",
    f"{len(expected_hours):,}"
)


coverage_rows = []


for city in CITIES.keys():

    city_data = (

        weather.loc[
            weather[
                "city"
            ]
            ==
            city,
            "time"
        ]

        .drop_duplicates()
    )


    missing_hours = (

        expected_hours

        .difference(
            pd.DatetimeIndex(
                city_data
            )
        )
    )


    coverage_rows.append({

        "city":
            city,

        "observed_hours":
            len(
                city_data
            ),

        "expected_hours":
            len(
                expected_hours
            ),

        "missing_hours":
            len(
                missing_hours
            ),

        "coverage_percent":
            (
                100
                *
                len(
                    city_data
                )
                /
                len(
                    expected_hours
                )
            )

    })


coverage = pd.DataFrame(
    coverage_rows
)


print("\n" + "=" * 115)
print("HOURLY COVERAGE AUDIT")
print("=" * 115)


display(
    coverage
)


# ======================================================================
# 15. BASIC PHYSICAL SANITY CHECKS
# ======================================================================

physical_checks = {

    "temperature_2m":
        (-5, 50),

    "relative_humidity_2m":
        (0, 100),

    "rain":
        (0, 300),

    "pressure_msl":
        (900, 1100),

    "cloud_cover":
        (0, 100),

    "wind_speed_100m":
        (0, 200),

    "wind_direction_100m":
        (0, 360),

    "soil_temperature_100_to_255cm":
        (0, 50)

}


physical_audit = []


for variable, limits in (
    physical_checks.items()
):

    lower, upper = limits


    invalid = (

        (
            weather[
                variable
            ]
            <
            lower
        )

        |

        (
            weather[
                variable
            ]
            >
            upper
        )

    )


    physical_audit.append({

        "variable":
            variable,

        "lower_bound":
            lower,

        "upper_bound":
            upper,

        "outside_range_count":
            int(
                invalid.sum()
            ),

        "outside_range_percent":
            float(
                invalid.mean()
                *
                100
            )

    })


physical_audit = pd.DataFrame(
    physical_audit
)


print("\n" + "=" * 115)
print("PHYSICAL SANITY AUDIT")
print("=" * 115)


display(
    physical_audit
)


# ======================================================================
# 16. SAVE
# ======================================================================

weather.to_csv(

    OUTPUT_FILE,

    index=False,

    date_format=
        "%Y-%m-%dT%H:%M"

)


print("\n" + "=" * 115)
print("ERA5 WEATHER DATA SAVED")
print("=" * 115)


print(
    OUTPUT_FILE
)


# ======================================================================
# 17. SAVE METADATA
# ======================================================================

metadata = {

    "source":
        "Open-Meteo Historical Weather API",

    "reanalysis_model":
        "ERA5",

    "timezone":
        "Asia/Dhaka",

    "download_start_date":
        START_DATE,

    "download_end_date":
        END_DATE,

    "cities":
        CITIES,

    "hourly_variables":
        HOURLY_VARIABLES,

    "rows":
        int(
            len(
                weather
            )
        )

}


METADATA_FILE = (

    WEATHER_RAW_DIR
    /
    "bangladesh_weather_era5_2015_2026_metadata.json"

)


with open(

    METADATA_FILE,

    "w"

) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )


# ======================================================================
# 18. FINAL VALIDATION
# ======================================================================

print("\n" + "=" * 115)
print("FINAL VALIDATION")
print("=" * 115)


print(
    "Start:",
    weather[
        "time"
    ].min()
)

print(
    "End:",
    weather[
        "time"
    ].max()
)

print(
    "Rows:",
    f"{len(weather):,}"
)

print(
    "Cities:",
    weather[
        "city"
    ].nunique()
)

print(
    "Duplicate city-hours:",
    duplicate_count
)

print(
    "Missing numerical cells:",
    int(
        weather[
            weather_variables
        ]
        .isna()
        .sum()
        .sum()
    )
)


print(
    "\nSaved CSV:"
)

print(
    OUTPUT_FILE
)


print(
    "\nSaved metadata:"
)

print(
    METADATA_FILE
)


print("\n" + "=" * 115)
print("STEP W0 COMPLETED")
print("=" * 115)

print(
    "\nNEXT:"
)

print(
    "Use this new ERA5 CSV as the W1 input, "
    "then rerun W1 and W2."
)

STEP W0: DOWNLOAD FULL-PERIOD ERA5 WEATHER DATA
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Download windows:
2015-04-01 -> 2015-12-31
2016-01-01 -> 2016-12-31
2017-01-01 -> 2017-12-31
2018-01-01 -> 2018-12-31
2019-01-01 -> 2019-12-31
2020-01-01 -> 2020-12-31
2021-01-01 -> 2021-12-31
2022-01-01 -> 2022-12-31
2023-01-01 -> 2023-12-31
2024-01-01 -> 2024-12-31
2025-01-01 -> 2025-12-31
2026-01-01 -> 2026-03-08

CITY: Dhaka

Rows received: 6,600

Rows received: 8,784

Rows received: 8,760

Rows received: 8,760

Rows received: 8,760

Rows received: 8,784

Rows received: 8,760

Rows received: 8,760

Rows received: 8,760

Rows received: 8,784

Rows received: 8,760

Rows received: 1,608

CITY: Chittagong

Rows received: 6,600

Rows received: 8,784

Rows received: 8,760

Rows received: 8,760

Rows received: 8,760

Rows received: 8,784

Rows received: 8,760

Rows received: 8,760

Rows received: 8,760

Rows recei

,missing_count,missing_percent
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
rain,0,0.0
pressure_msl,0,0.0
cloud_cover,0,0.0
wind_speed_100m,0,0.0
wind_direction_100m,0,0.0
soil_temperature_100_to_255cm,0,0.0



CITY COVERAGE


,city,rows,start,end,unique_hours
0,Chittagong,95880,2015-04-01,2026-03-08 23:00:00,95880
1,Dhaka,95880,2015-04-01,2026-03-08 23:00:00,95880
2,Rajshahi,95880,2015-04-01,2026-03-08 23:00:00,95880
3,Sylhet,95880,2015-04-01,2026-03-08 23:00:00,95880



Expected hours per city: 95,880

HOURLY COVERAGE AUDIT


,city,observed_hours,expected_hours,missing_hours,coverage_percent
0,Dhaka,95880,95880,0,100.0
1,Chittagong,95880,95880,0,100.0
2,Rajshahi,95880,95880,0,100.0
3,Sylhet,95880,95880,0,100.0



PHYSICAL SANITY AUDIT


,variable,lower_bound,upper_bound,outside_range_count,outside_range_percent
0,temperature_2m,-5,50,0,0.0
1,relative_humidity_2m,0,100,0,0.0
2,rain,0,300,0,0.0
3,pressure_msl,900,1100,0,0.0
4,cloud_cover,0,100,0,0.0
5,wind_speed_100m,0,200,0,0.0
6,wind_direction_100m,0,360,0,0.0
7,soil_temperature_100_to_255cm,0,50,0,0.0



ERA5 WEATHER DATA SAVED
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/raw/bangladesh_weather_era5_2015_2026.csv

FINAL VALIDATION
Start: 2015-04-01 00:00:00
End: 2026-03-08 23:00:00
Rows: 383,520
Cities: 4
Duplicate city-hours: 0
Missing numerical cells: 0

Saved CSV:
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/raw/bangladesh_weather_era5_2015_2026.csv

Saved metadata:
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/raw/bangladesh_weather_era5_2015_2026_metadata.json

STEP W0 COMPLETED

NEXT:
Use this new ERA5 CSV as the W1 input, then rerun W1 and W2.
